# Stockage vectoriel réel — persistance, index ANN et compromis rappel-exact

**Navigation** : [Version Python](05-Stockage-Vectoriel.ipynb) · [RAG et mémoire sémantique](README.md) · [GenAI](../README.md)

**Objectifs** : mesurer la recherche exacte, la persistance et le filtrage Qdrant, puis explorer un graphe petit-monde et le compromis rappel/latence.

**Prérequis** : .NET Interactive, accès NuGet et Docker opérationnel (image `qdrant/qdrant:v1.20.0`). Aucun GPU ni clé API. Durée indicative : 10 minutes. Le carnet crée un conteneur et un volume temporaires à lui ; l'exécuter jusqu'au nettoyage final.

Le SDK C# ne fournit pas le mode local embarqué Python. Ici Qdrant.Client 1.20.0 interroge un vrai serveur Rust 1.20.0 et force la recherche exacte. Le mode local Python est lui aussi exact mais constitue une autre implémentation ; les générateurs aléatoires et les temps diffèrent. Les vecteurs synthétiques isolent la recherche de la qualité des embeddings.

In [1]:
#r "nuget: Qdrant.Client, 1.20.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Text;
using System.IO;
using System.Net;
using System.Net.Sockets;
using System.Threading.Tasks;
using Microsoft.DotNet.Interactive.Formatting;
using Qdrant.Client;
using Qdrant.Client.Grpc;
Console.WriteLine($"Runtime .NET {Environment.Version} ; Qdrant.Client NuGet 1.20.0 (assembly {typeof(QdrantClient).Assembly.GetName().Version}) ; serveur requis 1.20.0");

Installed Packages Qdrant.Client, 1.20.0

Runtime .NET 10.0.14 ; Qdrant.Client NuGet 1.20.0 (assembly 1.0.0.0) ; serveur requis 1.20.0


## 1. La limite du k-NN exact — le mur de latence

Pour chaque taille, on calcule toutes les distances euclidiennes et on maintient les dix meilleures. Trois passages sont moyennés, en dimension 64 ; l'allocation du corpus est hors chronométrage.

In [2]:
float[][] RandomData(int n, int d, int seed) { var r=new Random(seed); return Enumerable.Range(0,n).Select(_=>Enumerable.Range(0,d).Select(_=>(float)r.NextDouble()).ToArray()).ToArray(); }
double Dist(float[] a,float[] b) { double s=0; for(int j=0;j<a.Length;j++){double x=a[j]-b[j];s+=x*x;} return s; }
int[] Exact(float[][] db,float[] q,int k) { var heap=new PriorityQueue<int,double>(); for(int i=0;i<db.Length;i++){double d=Dist(db[i],q);heap.Enqueue(i,-d);if(heap.Count>k)heap.Dequeue();} return heap.UnorderedItems.OrderBy(x=>-x.Priority).Select(x=>x.Element).ToArray(); }
(double n,double ms)[] Wall(int[] sizes,int d){var rows=new List<(double,double)>();foreach(int n in sizes){var db=RandomData(n,d,42);var q=RandomData(1,d,12)[0]; var sw=Stopwatch.StartNew();for(int j=0;j<3;j++)Exact(db,q,10);sw.Stop(); rows.Add((n,sw.Elapsed.TotalMilliseconds/3));Console.WriteLine($"N={n:N0}, d={d} : {sw.Elapsed.TotalMilliseconds/3:F2} ms, 3 passages");}return rows.ToArray();}
var wall=Wall(new[]{1000,10000,100000,1000000},64);

N=1,000, d=64 : 5.16 ms, 3 passages


N=10,000, d=64 : 5.78 ms, 3 passages


N=100,000, d=64 : 64.67 ms, 3 passages


N=1,000,000, d=64 : 272.00 ms, 3 passages


### Lecture du résultat

Les moyennes observées passent de 5,16 ms à N=1000 à 272,00 ms à N=1 000 000. Chaque requête effectue N distances ; les temps incluent la sélection et dépendent de la machine.

In [3]:
record SvgView(string Content);
Formatter.Register<SvgView>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
string Plot((double x,double y)[] a,string title,string xlab,string ylab,bool logX,bool logY,(double x,double y)[] b=null,string y2lab="") {
 double lx(double x)=>logX?Math.Log10(x):x; double ly(double y)=>logY?Math.Log10(Math.Max(y,1e-9)):y;
 double xmin=a.Min(p=>lx(p.x)),xmax=a.Max(p=>lx(p.x)),ymin=logY?a.Min(p=>ly(p.y)):0,ymax=a.Max(p=>ly(p.y))*1.1; if(b!=null){ymin=0;ymax=1;} if(ymax<=ymin)ymax=ymin+1;
 double X(double x)=>85+(lx(x)-xmin)/(xmax-xmin)*510; double Y(double y)=>340-(ly(y)-ymin)/(ymax-ymin)*280;
 var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='720' height='410' viewBox='0 0 720 410'><rect width='720' height='410' fill='white'/>");
 string esc(string s)=>System.Security.SecurityElement.Escape(s); void T(double x,double y,string s,string color="black")=>sb.Append($"<text x='{F(x)}' y='{F(y)}' fill='{color}' font-size='13'>{esc(s)}</text>");
 sb.Append("<path d='M85 60 V340 H595' fill='none' stroke='black'/>");T(110,25,title);T(260,395,xlab);T(4,48,ylab,"#1763a5");
 for(int i=0;i<5;i++){double tx=85+i*127.5,ty=340-i*70;sb.Append($"<path d='M{F(tx)} 340 v5 M80 {F(ty)} h5' stroke='black'/>");T(tx-15,365,F(logX?Math.Pow(10,xmin+(xmax-xmin)*i/4):xmin+(xmax-xmin)*i/4));T(5,ty+4,F(logY?Math.Pow(10,ymin+(ymax-ymin)*i/4):ymin+(ymax-ymin)*i/4));}
 void Line((double x,double y)[] points,Func<double,double> yy,string col){sb.Append("<polyline fill='none' stroke='"+col+"' stroke-width='2' points='");foreach(var p in points)sb.Append(F(X(p.x))+","+F(yy(p.y))+" ");sb.Append("'/>");foreach(var p in points)sb.Append($"<circle cx='{F(X(p.x))}' cy='{F(yy(p.y))}' r='3' fill='{col}'/>");}
 Line(a,Y,"#1763a5");if(b!=null){sb.Append("<path d='M85 60 H595' stroke='#555' stroke-dasharray='5 4'/>");T(430,54,"exact : rappel 1", "#555");double max=b.Max(p=>p.y)*1.1; if(max==0)max=1;Line(b,y=>340-y/max*280,"#b3261e");sb.Append("<path d='M595 60 V340' stroke='#b3261e'/>");T(600,48,y2lab,"#b3261e");for(int i=0;i<5;i++)T(600,344-i*70,F(max*i/4),"#b3261e");}sb.Append("</svg>");return sb.ToString();
}
var svgWall=Plot(wall,"k-NN exact","N (log)","ms (log)",true,true); display(new SvgView(svgWall)); Console.WriteLine("Figure : latence moyenne en ms, axes logarithmiques.");

k-NN exact N (log) ms (log) 1000 5.157 5623.413 15.989 31622.777 49.571 177827.941 153.681 1000000 476.451

Figure : latence moyenne en ms, axes logarithmiques.


### Lecture du graphique

La montée traduit le balayage du corpus ; les petits N peuvent être dominés par les coûts fixes. Une droite en log-log indique une loi de puissance, sans établir un seuil universel.

### Exemple guidé 1 — dimension 128

On reprend les trois passages sur 10 000 et 100 000 vecteurs.

In [4]:
var guide128=Wall(new[]{10000,100000},128); Console.WriteLine("Exemple guidé : d=128, moyenne de trois passages par taille.");

N=10,000, d=128 : 4.77 ms, 3 passages


N=100,000, d=128 : 152.15 ms, 3 passages


Exemple guidé : d=128, moyenne de trois passages par taille.


### Exercice 1 — dimension

À N fixé, mesurer les dimensions 32, 64, 128 et 256 sur trois passages. Compléter le parcours des dimensions en appelant la fonction de mesure fournie. Une fixture déterministe vérifie que chaque dimension est bien transmise ; elle ne certifie pas les performances.

In [5]:
Dictionary<int,double> LatenceDimensions(Func<int,double> mesurer){ // TODO : appeler mesurer pour 32, 64, 128 et 256
 return null;
}
var trace1=new List<int>();var trace2=new List<int>();
var f1=LatenceDimensions(d=>{trace1.Add(d);return d/4.0;});
if(f1==null)Console.WriteLine("Exercice 1 à compléter.");
else {
 var f2=LatenceDimensions(d=>{trace2.Add(d);return 1000.0/d;});
 int[] dims={32,64,128,256};
 bool ok=f2!=null && f1.Keys.OrderBy(x=>x).SequenceEqual(dims) && f2.Keys.OrderBy(x=>x).SequenceEqual(dims) && trace1.OrderBy(x=>x).SequenceEqual(dims) && trace2.OrderBy(x=>x).SequenceEqual(dims) && dims.All(d=>double.IsFinite(f1[d]) && double.IsFinite(f2[d]) && Math.Abs(f1[d]-d/4.0)<1e-9 && Math.Abs(f2[d]-1000.0/d)<1e-9);
 if(!ok)Console.WriteLine("Fixtures incorrectes : appeler chaque dimension et retourner sa mesure.");
 else {var resultat=LatenceDimensions(d=>Wall(new[]{10000},d)[0].ms);if(resultat==null || !resultat.Keys.OrderBy(x=>x).SequenceEqual(dims) || resultat.Values.Any(x=>!double.IsFinite(x)||x<=0))Console.WriteLine("Résultat invalide.");else Console.WriteLine("Mesures : "+string.Join(", ",resultat.Select(kv=>$"d={kv.Key} : {kv.Value:F2} ms")));}
}

Exercice 1 à compléter.


## 2. Une persistance réelle — Qdrant serveur

On crée un volume Docker propre au carnet et un port gRPC local choisi par le système. Le client ferme sa connexion, le serveur redémarre, puis le même compte et les mêmes voisins avec leurs charges utiles sont vérifiés sans réinsertion.

In [6]:
string RunDocker(params string[] args){var psi=new ProcessStartInfo("docker"){RedirectStandardOutput=true,RedirectStandardError=true};foreach(var a in args)psi.ArgumentList.Add(a);using var p=Process.Start(psi);var o=p.StandardOutput.ReadToEndAsync();var e=p.StandardError.ReadToEndAsync();if(!p.WaitForExit(60000)){p.Kill(true);throw new Exception("Docker : délai dépassé");}Task.WaitAll(o,e);if(p.ExitCode!=0)throw new Exception("Docker : "+e.Result);return o.Result.Trim();}
var listener=new TcpListener(IPAddress.Loopback,0);listener.Start();int port=((IPEndPoint)listener.LocalEndpoint).Port;listener.Stop();
string suffix=Guid.NewGuid().ToString("N");string container="rag05-"+suffix,volume="rag05-"+suffix;
void Cleanup(){try{RunDocker("rm","-f",container);}catch(Exception ex){Console.WriteLine("Nettoyage conteneur : "+ex.Message);}try{RunDocker("volume","rm",volume);}catch(Exception ex){Console.WriteLine("Nettoyage volume : "+ex.Message);}}
async Task<QdrantClient> Connect()
{
    using var budget = new System.Threading.CancellationTokenSource(TimeSpan.FromSeconds(30));
    for (int i = 0; i < 30 && !budget.IsCancellationRequested; i++)
    {
        var connection = new QdrantClient("127.0.0.1", port, grpcTimeout: TimeSpan.FromSeconds(10));
        try { await connection.ListCollectionsAsync(budget.Token); return connection; }
        catch { connection.Dispose(); if (budget.IsCancellationRequested) break; }
        try { await Task.Delay(1000, budget.Token); }
        catch (OperationCanceledException) { break; }
    }
    throw new TimeoutException("Qdrant indisponible après le budget global de 30 secondes.");
}
Console.WriteLine($"Ressources propres au carnet : conteneur {container}, volume {volume}. En cas d'interruption : docker rm -f {container} puis docker volume rm {volume}.");
QdrantClient client=null;try{RunDocker("volume","create",volume);RunDocker("run","-d","--name",container,"-p",$"127.0.0.1:{port}:6334","-v",$"{volume}:/qdrant/storage","qdrant/qdrant:v1.20.0");client=await Connect();}catch{Cleanup();throw;}
Console.WriteLine($"Serveur Qdrant 1.20.0 démarré sur le port local {port}, volume dédié.");

Ressources propres au carnet : conteneur rag05-01d460b57ee74921848aff47687e8e96, volume rag05-01d460b57ee74921848aff47687e8e96. En cas d'interruption : docker rm -f rag05-01d460b57ee74921848aff47687e8e96 puis docker volume rm rag05-01d460b57ee74921848aff47687e8e96.


Serveur Qdrant 1.20.0 démarré sur le port local 45657, volume dédié.


### Corpus synthétique

Les cinq thèmes correspondent à cinq centres ; les trois sources et dates sont cycliques. Les lots de 500 évitent une insertion monolithique.

In [7]:
string[] themes={"agent","approvisionnement","sécurité","réseau","données"},sources={"conversation","code","note"},dates={"2026-01-10","2026-02-11","2026-03-12"};
double Normal(Random r){double u1=1-r.NextDouble(),u2=1-r.NextDouble();return Math.Sqrt(-2*Math.Log(u1))*Math.Cos(2*Math.PI*u2);}
float[][] Cluster(int n,int d,int count,int seed){var r=new Random(seed);var centers=RandomData(count,d,seed+1);return Enumerable.Range(0,n).Select(i=>{int c=r.Next(count);return centers[c].Select(v=>(float)(v*2+Normal(r)*0.25)).ToArray();}).ToArray();}
var centers=RandomData(5,64,5);var rng=new Random(1);var vecs=new float[2000][];var labels=new int[2000];for(int i=0;i<2000;i++){labels[i]=rng.Next(5);vecs[i]=centers[labels[i]].Select(x=>(float)(x*2+Normal(rng)*0.25)).ToArray();}
await client.CreateCollectionAsync("chunks",new VectorParams{Size=64,Distance=Distance.Cosine});
for(int start=0;start<2000;start+=500){var points=Enumerable.Range(start,500).Select(i=>new PointStruct{Id=(ulong)(i+1),Vectors=vecs[i],Payload={{"source",sources[i%3]},{"theme",themes[labels[i]]},{"date",dates[i%3]}}}).ToArray();await client.UpsertAsync("chunks",points);}
var query=Enumerable.Range(0,64).Select(j=>(float)vecs.Average(v=>v[j])).ToArray();var before=await client.QueryAsync("chunks",query:query,searchParams:new SearchParams{Exact=true},limit:10);var count=await client.CountAsync("chunks",exact:true);Console.WriteLine($"{count} points, d=64, {labels.Distinct().Count()} thèmes, 3 sources, 3 dates ; 4 lots de 500 ; premier voisin {before[0].Id.Num}.");

2000 points, d=64, 5 thèmes, 3 sources, 3 dates ; 4 lots de 500 ; premier voisin 1723.


### Lecture du résultat

Les quatre lots ont produit 2000 points, cinq thèmes, trois sources et trois dates ; le premier voisin du centroïde est le point 1723. Le compte seul ne prouve pas la persistance.

In [8]:
client.Dispose();RunDocker("restart",container);client=await Connect();var after=await client.QueryAsync("chunks",query:query,searchParams:new SearchParams{Exact=true},limit:10);var afterCount=await client.CountAsync("chunks",exact:true);
if(afterCount!=2000 || !before.Select(x=>x.Id.Num).SequenceEqual(after.Select(x=>x.Id.Num)) || !before.Zip(after).All(z=>z.First.Payload["source"].StringValue==z.Second.Payload["source"].StringValue && z.First.Payload["theme"].StringValue==z.Second.Payload["theme"].StringValue && z.First.Payload["date"].StringValue==z.Second.Payload["date"].StringValue))throw new Exception("Persistance non vérifiée");
Console.WriteLine($"Après fermeture, redémarrage et réouverture : {afterCount} points ; 10 voisins et payloads identiques.");

Après fermeture, redémarrage et réouverture : 2000 points ; 10 voisins et payloads identiques.


### Lecture du résultat

Après redémarrage, le compte est toujours 2000 et les dix voisins ainsi que leurs charges utiles restent identiques, sans nouvelle insertion.

## 3. HNSW déplié — mini-graphe

Six points grossiers et trente points fins possèdent respectivement deux et trois liens proches. On compare dix descentes gourmandes à la vérité exacte.

In [9]:
int[][] NearGraph(float[][] pts,int m)=>Enumerable.Range(0,pts.Length).Select(i=>Enumerable.Range(0,pts.Length).Where(j=>i!=j).OrderBy(j=>Dist(pts[i],pts[j])).Take(m).ToArray()).ToArray();
(int node,int evals) Greedy(float[] q,float[][] pts,int[][] adj,int entry){int cur=entry,ev=0;double best=Dist(pts[cur],q);bool changed;do{changed=false;foreach(int nb in adj[cur]){double d=Dist(pts[nb],q);ev++;if(d<best){cur=nb;best=d;changed=true;}}}while(changed);return(cur,ev);}
var coarse=RandomData(6,2,3);var fine=RandomData(30,2,4);var ca=NearGraph(coarse,2);var fa=NearGraph(fine,3);int hits=0,total=0;foreach(var q in RandomData(10,2,8)){int entry=Exact(coarse,q,1)[0];var c=Greedy(q,coarse,ca,entry);int start=Exact(fine,coarse[c.node],1)[0];var f=Greedy(q,fine,fa,start);total+=c.evals+f.evals;if(f.node==Exact(fine,q,1)[0])hits++;}Console.WriteLine($"Vrais voisins atteints : {hits}/10 ; descentes : {total/10.0:F1} distances/requête ; sélection des entrées : 36 et initialisation des descentes : 2 distances supplémentaires ; exact fin : 30.");

Vrais voisins atteints : 5/10 ; descentes : 7.7 distances/requête ; sélection des entrées : 36 et initialisation des descentes : 2 distances supplémentaires ; exact fin : 30.


### Lecture du résultat

La descente atteint le vrai voisin 5 fois sur 10 avec 7,7 distances dans son cœur. Il faut ajouter 36 distances de sélection et deux distances initiales : soit 45,7 au total, contre 30 distances pour la référence exacte fine. Ce compteur partiel ne prouve donc pas un gain global.

## 4. Compromis mesuré — graphe petit-monde

Le graphe jouet comporte 10 000 vecteurs de dimension 64, 60 grappes, huit liens courts choisis dans un pool de 500 et trois liens longs. Il illustre le mécanisme sans reproduire l'index HNSW complet du serveur.

In [10]:
var data=Cluster(10000,64,60,9);var random=new Random(7);var adj=new int[data.Length][];var build=Stopwatch.StartNew();for(int i=0;i<data.Length;i++){var pool=new HashSet<int>();while(pool.Count<500)pool.Add(random.Next(data.Length));var near=pool.OrderBy(j=>Dist(data[i],data[j])).Take(8);var longLinks=new HashSet<int>();while(longLinks.Count<3){int j=random.Next(data.Length);if(j!=i)longLinks.Add(j);}adj[i]=near.Concat(longLinks).ToArray();}build.Stop();Console.WriteLine($"Graphe : {data.Length} nœuds, pool 500, m=8, r=3 ; construction {build.Elapsed.TotalSeconds:F2} s.");

Graphe : 10000 nœuds, pool 500, m=8, r=3 ; construction 4.00 s.


### Lecture du résultat

La construction du graphe a pris 4,01 s. Elle précède les mesures et son coût n’est pas inclus dans les latences de recherche.

In [11]:
var queries=RandomData(20,64,11).Select(q=>q.Select(x=>x*2).ToArray()).ToArray();var gt=queries.Select(q=>Exact(data,q,10).ToHashSet()).ToArray();var exactWatch=Stopwatch.StartNew();foreach(var q in queries)Exact(data,q,10);exactWatch.Stop();Console.WriteLine($"Référence exacte : rappel 1, {exactWatch.Elapsed.TotalMilliseconds/20:F2} ms et 10000 distances par requête.");

Référence exacte : rappel 1, 5.82 ms et 10000 distances par requête.


### Lecture du résultat

Le balayage exact effectue 10 000 distances et prend 2,05 ms par requête dans cette exécution. La latence ne comprend pas la construction du graphe.

In [12]:
(int[] ids,int evals) Beam(float[] q,int ef,int k){int entry=Enumerable.Range(0,200).OrderBy(i=>Dist(data[i],q)).First();var cand=new PriorityQueue<int,double>();var best=new PriorityQueue<int,double>();var seen=new HashSet<int>{entry};double d0=Dist(data[entry],q);int ev=1;cand.Enqueue(entry,d0);best.Enqueue(entry,-d0);while(cand.Count>0){cand.TryDequeue(out int node,out double d);best.TryPeek(out _,out double worst);if(best.Count>=ef && d> -worst)break;foreach(int nb in adj[node]){if(!seen.Add(nb))continue;double dn=Dist(data[nb],q);ev++;best.TryPeek(out _,out worst);if(best.Count<ef || dn< -worst){best.Enqueue(nb,-dn);cand.Enqueue(nb,dn);if(best.Count>ef)best.Dequeue();}}}return(best.UnorderedItems.OrderBy(x=>-x.Priority).Take(k).Select(x=>x.Element).ToArray(),ev);}
(double ef,double recall,double ms,double core)[] Measure(int[] efs,int k){var rows=new List<(double,double,double,double)>();foreach(int ef in efs){double recall=0,ms=0,eval=0;for(int i=0;i<queries.Length;i++){var watch=Stopwatch.StartNew();var result=Beam(queries[i],ef,k);watch.Stop();ms+=watch.Elapsed.TotalMilliseconds;var truth=k==10?gt[i]:Exact(data,queries[i],k).ToHashSet();recall+=result.ids.Count(truth.Contains)/(double)k;eval+=result.evals;}rows.Add((ef,recall/20,ms/20,eval/20));Console.WriteLine($"ef={ef,3} rappel@{k}={recall/20:F3}, {ms/20:F2} ms, cœur={eval/20:F1}, total={eval/20+200:F1} distances");}return rows.ToArray();}
var rowsEf=Measure(new[]{1,2,4,8,16,32,64},10);

ef=  1 rappel@10=0.030, 0.48 ms, cœur=16.6, total=216.6 distances


ef=  2 rappel@10=0.060, 0.31 ms, cœur=38.0, total=238.1 distances


ef=  4 rappel@10=0.105, 0.31 ms, cœur=61.5, total=261.6 distances


ef=  8 rappel@10=0.195, 0.18 ms, cœur=103.5, total=303.6 distances


ef= 16 rappel@10=0.365, 0.20 ms, cœur=184.8, total=384.8 distances


ef= 32 rappel@10=0.475, 0.31 ms, cœur=292.1, total=492.1 distances


ef= 64 rappel@10=0.660, 0.60 ms, cœur=506.4, total=706.4 distances


### Lecture du résultat

Le rappel@10 observé passe de 0,030 à ef=1 à 0,660 à ef=64 ; à ef=32, le cœur compte 292,1 distances et le total 492,1 après ajout des 200 du pool initial. La distance du point d’entrée est recomptée dans le cœur. Pour `ef < k`, au plus `ef` résultats plafonnent le rappel à `ef/k` : limite de cardinalité distincte de la qualité du parcours. Ni rappel ni latence ne sont garantis monotones.

In [13]:
var svgTrade=Plot(rowsEf.Select(r=>(r.ef,r.recall)).ToArray(),"Rappel et latence selon ef","ef (log)","rappel@10",true,false,rowsEf.Select(r=>(r.ef,r.ms)).ToArray(),"latence (ms)");display(new SvgView(svgTrade));Console.WriteLine("Bleu : rappel@10 (axe gauche) ; rouge : latence en ms (axe droit).");

Rappel et latence selon ef ef (log) rappel@10 1 0 2.828 0.25 8 0.5 22.627 0.75 64 1 exact : rappel 1 latence (ms) 0 0.165 0.33 0.494 0.659

Bleu : rappel@10 (axe gauche) ; rouge : latence en ms (axe droit).


### Lecture du graphique

Les deux axes représentent des unités différentes : comparer les tendances, pas la hauteur des courbes. Ces résultats à N=10 000 ne mesurent pas une latence ANN à un million de points.

### Exemple guidé 2 — prix du rappel

On répète les vingt requêtes pour `ef` égal à 32, 64 et 128.

In [14]:
var guideEf=Measure(new[]{32,64,128},10);Console.WriteLine("Exemple guidé : rappel et coût du cœur, pool initial à ajouter.");

ef= 32 rappel@10=0.475, 0.27 ms, cœur=292.1, total=492.1 distances


ef= 64 rappel@10=0.660, 0.46 ms, cœur=506.4, total=706.4 distances


ef=128 rappel@10=0.885, 0.73 ms, cœur=876.7, total=1076.7 distances


Exemple guidé : rappel et coût du cœur, pool initial à ajouter.


### Exercice 2 — rappel selon k

Compléter le calcul des intersections moyennes pour k=5, 10 et 20 ; la fixture déterministe vérifie les trois valeurs avant l'application aux vingt requêtes avec `ef=32`.

In [15]:
Dictionary<int,double> RappelParK(int[][] truth,int[][] found){ // TODO : moyenne des intersections aux préfixes k=5,10,20
 return null;
}
int[] ks={5,10,20};
Dictionary<int,double> Oracle(int[][] t,int[][] f)=>ks.ToDictionary(k=>k,k=>Enumerable.Range(0,t.Length).Average(i=>t[i].Take(k).Intersect(f[i].Take(k)).Count()/(double)k));
bool Correct(Dictionary<int,double> actual,Dictionary<int,double> expected)=>actual!=null && actual.Keys.OrderBy(x=>x).SequenceEqual(ks) && ks.All(k=>double.IsFinite(actual[k])&&Math.Abs(actual[k]-expected[k])<1e-9);
var fixtureTruth=new[]{Enumerable.Range(0,20).ToArray(),Enumerable.Range(0,20).ToArray()};var fixtureFound=new[]{Enumerable.Range(0,20).ToArray(),Enumerable.Range(10,20).ToArray()};
var f1R=RappelParK(fixtureTruth,fixtureFound);
if(f1R==null)Console.WriteLine("Exercice 2 à compléter.");else {
 var t2=new[]{Enumerable.Range(0,20).ToArray(),Enumerable.Range(20,20).ToArray()};var a2=new[]{Enumerable.Range(40,20).ToArray(),Enumerable.Range(20,20).Reverse().ToArray()};var f2R=RappelParK(t2,a2);
 if(!Correct(f1R,Oracle(fixtureTruth,fixtureFound))||!Correct(f2R,Oracle(t2,a2)))Console.WriteLine("Fixtures incorrectes : vérifiez les intersections et le dénominateur.");
 else {var truth=queries.Select(q=>Exact(data,q,20)).ToArray();var found=queries.Select(q=>Beam(q,32,20).ids).ToArray();var result=RappelParK(truth,found);if(!Correct(result,Oracle(truth,found)))Console.WriteLine("Résultat incorrect sur les requêtes réelles.");else Console.WriteLine("Rappel : "+string.Join(", ",result.Select(x=>$"k={x.Key} : {x.Value:F3}")));}
}

Exercice 2 à compléter.


## 5. Métadonnées et filtrage Qdrant

Le filtre natif sélectionne les points de source `code` avant de renvoyer les voisins exacts. Cette requête ne démontre pas un parcours HNSW filtré.

In [16]:
Condition Eq(string key,string value)=>new Condition{Field=new FieldCondition{Key=key,Match=new Match{Keyword=value}}};
var codeFilter=new Filter{Must={Eq("source","code")}};var unfiltered=await client.QueryAsync("chunks",query:query,searchParams:new SearchParams{Exact=true},limit:10);var filtered=await client.QueryAsync("chunks",query:query,filter:codeFilter,searchParams:new SearchParams{Exact=true},limit:10);if(filtered.Any(x=>x.Payload["source"].StringValue!="code"))throw new Exception("Filtre incorrect");Console.WriteLine($"Sans filtre : {unfiltered.Count} voisins, thèmes = {string.Join(", ",unfiltered.Select(x=>x.Payload["theme"].StringValue))} ; source=code : {filtered.Count} voisins, thèmes = {string.Join(", ",filtered.Select(x=>x.Payload["theme"].StringValue))}, tous conformes ; corpus filtré : {await client.CountAsync("chunks",filter:codeFilter,exact:true)}.");

Sans filtre : 10 voisins, thèmes = données, données, données, données, données, données, données, données, données, données ; source=code : 10 voisins, thèmes = données, données, données, données, données, données, données, données, données, données, tous conformes ; corpus filtré : 667.


### Lecture du résultat

Les dix voisins filtrés satisfont la condition `source=code`, et 667 points sur 2000 satisfont cette condition dans tout le corpus. Dans cette exécution les voisins du centroïde ont tous le thème données, filtrés ou non ; cela dépend des tirages.

### Exemple guidé 3 — thème ET date

Deux conditions dans `Must` composent une conjonction.

In [17]:
var both=new Filter{Must={Eq("theme","sécurité"),Eq("date","2026-02-11")}};var bothHits=await client.QueryAsync("chunks",query:query,filter:both,searchParams:new SearchParams{Exact=true},limit:10);if(bothHits.Any(x=>x.Payload["theme"].StringValue!="sécurité"||x.Payload["date"].StringValue!="2026-02-11"))throw new Exception("Conjonction incorrecte");Console.WriteLine($"Thème sécurité ET date 2026-02-11 : {bothHits.Count} voisins, {await client.CountAsync("chunks",filter:both,exact:true)} points dans le corpus.");

Thème sécurité ET date 2026-02-11 : 10 voisins, 127 points dans le corpus.


### Lecture du résultat

La conjonction retient 127 points sur toute la collection ; la recherche renvoie dix voisins, bornés par sa limite.

### Exercice 3 — MatchAny et décompte

Construire un filtre sur les thèmes indiqués, puis retourner les voisins exacts et le nombre filtré et total. Deux jeux de thèmes sont contrôlés contre les étiquettes du corpus. Une requête top-k ne permet pas de déduire la proportion du corpus.

In [18]:
async Task<(IReadOnlyList<ScoredPoint> points,ulong filtered,ulong total)?> MatchAnyExercice(string[] selected){ // TODO : MatchAny, recherche exacte et comptes filtré/total
 await Task.CompletedTask; return null;
}
var fixtureFilter=await MatchAnyExercice(new[]{"sécurité","réseau"});
if(fixtureFilter==null)Console.WriteLine("Exercice 3 à compléter.");
else {
 async Task<bool> Check(string[] selected,(IReadOnlyList<ScoredPoint> points,ulong filtered,ulong total)? answer){
  if(answer==null)return false;var a=answer.Value;var referenceFilter=new Filter();foreach(var theme in selected)referenceFilter.Should.Add(Eq("theme",theme));
  var reference=await client.QueryAsync("chunks",query:query,filter:referenceFilter,searchParams:new SearchParams{Exact=true},limit:10);var countReference=await client.CountAsync("chunks",filter:referenceFilter,exact:true);
  return a.total==2000 && a.filtered==countReference && a.points!=null && a.points.Count==reference.Count && a.points.Zip(reference).All(z=>z.First.Id.Equals(z.Second.Id) && Math.Abs(z.First.Score-z.Second.Score)<1e-6 && selected.Contains(z.First.Payload["theme"].StringValue));
 }
 var second=await MatchAnyExercice(new[]{"agent","données"});
 if(!await Check(new[]{"sécurité","réseau"},fixtureFilter) || !await Check(new[]{"agent","données"},second))Console.WriteLine("Fixture incorrecte : vérifier voisins exacts et décomptes.");
 else Console.WriteLine($"Sécurité ou réseau : {fixtureFilter.Value.filtered}/{fixtureFilter.Value.total} points ; voisins exacts conformes.");
}

Exercice 3 à compléter.


## 6. Guide de choix

- **Petit corpus** : commencer par l'exact en mémoire, puis vérifier la latence sur le matériel cible.
- **Persistance locale Python** : le client local conserve les données et filtre exactement, sans serveur.
- **Serveur et HNSW** : Qdrant serveur permet un index ANN ; ici les requêtes serveur sont explicitement exactes. Mesurer sur le corpus cible avant de choisir l'index.
- **Fiabilité** : un `ef` plus large peut améliorer le rappel, à mesurer pour les requêtes réelles.
- **Débit** : un `ef` modéré peut réduire le travail, au prix éventuel de rappel ; observer les temps et le coût total.

## Exercices

- [Dimension](#exercice-1--dimension) : comparer 32, 64, 128 et 256 avec trois mesures.
- [Rappel selon k](#exercice-2--rappel-selon-k) : comparer k=5, 10 et 20 à ef=32.
- [MatchAny et décompte](#exercice-3--matchany-et-décompte) : rechercher deux thèmes et compter le corpus filtré et total.

La dernière cellule ferme le client et supprime uniquement les ressources temporaires de ce carnet.

In [19]:
client.Dispose();Cleanup();Console.WriteLine("Client fermé ; nettoyage du conteneur et du volume demandés.");

Client fermé ; nettoyage du conteneur et du volume demandés.


## À retenir

- L'exact calcule N distances ; sa latence est mesurée et dépend de la machine.
- Le serveur Qdrant conserve les points et charges utiles après redémarrage ; les requêtes utilisées ici sont exactes.
- Le graphe jouet illustre le compromis rappel/coût. Compter le pool initial et distinguer cardinalité et qualité du parcours.

## Vérifiez votre compréhension

<details><summary>Pourquoi un top-10 filtré de taille 10 ne signifie-t-il pas que tout le corpus est retenu ?</summary>La limite borne les résultats ; il faut un décompte filtré et un décompte total.</details>

<details><summary>Pourquoi ef=2 plafonne-t-il le rappel@10 à 0,2 ici ?</summary>Le beam conserve au plus deux résultats, indépendamment de la qualité des chemins.</details>

## Pour aller plus loin

Mesurer le serveur en mode ANN sur un corpus réel et vérifier rappel, latence et coût d'indexation à la taille visée. Voir la [documentation Qdrant](https://qdrant.tech/documentation/).